# v2.3.3 국회도서관 자료 정렬 실습

논문 검색 결과를 **학위논문 → 학술기사 → 도서자료** 순서로 정리하는 초보자용 Jupyter Notebook입니다.

- 공개 API 인증키나 인터넷 연결 없이 실행할 수 있습니다.
- 모든 실습 코드 다음에 **별도의 `assert` 검증 셀**을 제공합니다.
- 이 Notebook을 프로그램의 `app.py`와 같은 폴더에 두고 실행하세요.
- 국회도서관 사이트의 261건은 화면에서 본 수치입니다. 이 실습의 샘플은 실제 조회 결과가 아닙니다.


## 실습 1. 자료유형별 순서 정렬

In [ ]:
import pandas as pd
from nanet_search_flow import sort_catalog_records

# 연습용 가상 검색 결과: 원래 API에서 받은 순서가 섞여 있다고 가정합니다.
검색자료 = pd.DataFrame([
    {"DB":"일반도서", "controlno":"BOOK001", "자료명/저자사항":"데이터베이스 포렌식 입문"},
    {"DB":"국내학술기사", "controlno":"ART001", "자료명/저자사항":"포렌식 기술 동향"},
    {"DB":"학위논문", "controlno":"THESIS001", "자료명/저자사항":"데이터베이스 포렌식 연구"},
    {"DB":"일반도서", "controlno":"BOOK002", "자료명/저자사항":"디지털 포렌식 교재"},
    {"DB":"학위논문", "controlno":"THESIS002", "자료명/저자사항":"클라우드 DB 포렌식 연구"},
])
정렬자료 = sort_catalog_records(검색자료)
print(정렬자료[["자료분류", "자료명/저자사항"]].to_string(index=False))

In [ ]:
# 검증 셀 1: 학위논문 2건 → 학술기사 1건 → 도서자료 2건 순서인지 확인
assert 정렬자료["자료분류"].tolist() == [
    "학위논문", "학위논문", "학술기사", "도서자료", "도서자료"
]
assert 검색자료.iloc[0]["DB"] == "일반도서"  # 원본의 첫 행은 바뀌지 않음
print("검증 1 성공: 우선순위와 원본 보존")

## 실습 2. 다음 검색 페이지의 자료 추가

In [ ]:
from nanet_search_flow import merge_catalog_pages

# 1~2페이지에서 얻은 자료와 3페이지의 추가 자료를 합칩니다.
기존결과 = 검색자료.iloc[:3].copy()
추가결과 = pd.concat([
    검색자료.iloc[[3, 4]],
    검색자료.iloc[[0]],  # 첫 번째 도서가 중복으로 왔다고 가정
], ignore_index=True)
논문목록, 확인대기, 도서목록, 전체정렬 = merge_catalog_pages(기존결과, 추가결과)
print("학위논문·학술기사:", len(논문목록), "건")
print("도서자료:", len(도서목록), "건")
print("전체:", len(전체정렬), "건")

In [ ]:
# 검증 셀 2: 중복 관리번호 제거, 유형별 정렬
assert len(전체정렬) == 5
assert len(논문목록) == 3
assert len(도서목록) == 2
assert 전체정렬.iloc[0]["자료분류"] == "학위논문"
print("검증 2 성공: 중복 제거와 페이지 병합")

## 실습 3. 자료별 정식 참고문헌

In [ ]:
from citation_styles import format_reference, suggested_citation

# 확인 가능한 정보만 활용해 참고문헌을 작성합니다.
예시도서 = {
    "DB": "일반도서", "자료명/저자사항": "데이터베이스 포렌식 / 김연구",
    "발행자": "서울 : 포렌식출판사, 2020", "발행년": "2020",
}
서지정보 = suggested_citation(예시도서)
참고문헌 = format_reference(서지정보)
print(참고문헌)

In [ ]:
# 검증 셀 3: 도서가 학위논문으로 잘못 인식되지 않고 정식 서지형식을 사용
assert 서지정보["type"] == "단행본"
assert "『데이터베이스 포렌식』" in 참고문헌
assert "포렌식출판사" in 참고문헌
assert "학위논문" not in 참고문헌
print("검증 3 성공: 도서자료 참고문헌 분리")

## 실습 4. API 전체 건수와 현재 수집 건수 구분

In [ ]:
# 실제 국회도서관 웹사이트를 조회하는 코드가 아닙니다.
# 사용자가 확인한 사이트의 검색 결과 261건을 가정해 계산을 연습합니다.
API_전체건수 = 261
현재_수집건수 = 40
추가_확인필요 = API_전체건수 - 현재_수집건수
print(f"전체 검색 결과: {API_전체건수}건")
print(f"현재 앱에 수집된 자료: {현재_수집건수}건")
print(f"추가로 확인할 수 있는 자료: 최대 {추가_확인필요}건")

In [ ]:
# 검증 셀 4: 분리된 건수가 올바른지 확인
assert 추가_확인필요 == 221
assert 현재_수집건수 < API_전체건수
print("검증 4 성공: 수집 범위에 따른 정렬 한계 인식")